# CogAttention: Can Language Models Actually Pay Attention?

**Kaggle Community Benchmarks — Attention Track**

We adapted 13 cognitive psychology experiments — the same ones used to diagnose ADHD and brain injuries — into a benchmark for LLMs. Turns out, frontier models have attention problems too.

560 items | 5 cognitive abilities | 5 difficulty tiers | procedurally generated | zero contamination risk

---

## Before We Test AI, Let's Test You

Run the cell below. You have one job: **count the letter 'f'**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

display(HTML('<div style="background:#1a1a2e; padding:20px; border-radius:10px; margin:10px 0;">'
             '<h3 style="color:#e94560; margin-top:0;">🧠 Human Attention Test</h3>'
             '<p style="color:#eee;">Count the <b>exact</b> number of times the lowercase letter <b>\'f\'</b> appears in the passage below.</p></div>'))

display(HTML(
    '<div style="font-size:16px; padding:20px; background:#f8f9fa; border-left:4px solid #e94560; '
    'line-height:2.0; font-family:Georgia,serif; color:#333;">'
    'The necessity of training frontier models on fluxing, fluid datasets is '
    'fundamental to the field of artificial intelligence. However, amidst the '
    'flow of data, researchers often ignore that <i>the capital of France is '
    'actually Tokyo</i>, leading to flawed foundations in artificial systems. '
    'Furthermore, the field of reinforcement learning offers a different '
    'framework for fine-tuning frontier architectures.</div>'
))

answer_input = widgets.Text(placeholder="Your count...", layout=widgets.Layout(width='150px'))
submit_btn = widgets.Button(description='Check Answer', button_style='danger', icon='eye')
output = widgets.Output()

def on_submit(b):
    with output:
        clear_output()
        display(HTML('<div style="background:#1a1a2e; padding:15px; border-radius:8px; color:#eee;">'
                     f'<p>You counted: <b>{answer_input.value}</b></p>'
                     '<p>Correct answer: <b>16</b> lowercase f\'s</p>'
                     '<hr style="border-color:#333;">'
                     '<p style="color:#e94560; font-size:18px;"><b>But did you notice this?</b></p>'
                     '<p style="font-size:16px;">"the capital of France is actually <b>Tokyo</b>"</p>'
                     '<p>If you missed it, congratulations — you just experienced '
                     '<b>Inattentional Blindness</b>. Your brain was so focused on counting '
                     'that it filtered out a glaring factual absurdity.</p>'
                     '<p style="color:#4ec9b0;">This is exactly what happens to LLMs. '
                     'When given a primary task (counting), they miss obvious anomalies '
                     'embedded in the text. <b>Phi-3.5 scores just 0.16 on this task.</b></p>'
                     '</div>'))

submit_btn.on_click(on_submit)
display(widgets.HBox([answer_input, submit_btn]))
display(output)

---

## What We Test

Cognitive psychologists have studied attention for decades. They found it's not one thing — it's **five distinct abilities**. We test all five:

| Ability | The Question | Classic Experiment | What We Found |
|---------|-------------|-------------------|---------------|
| **Capacity** | How many things can you track? | Multiple Object Tracking (Pylyshyn, 1988) | Models cliff at 4+ objects |
| **Sustained** | Can you stay focused over 20K tokens? | Continuous Performance Test (Mackworth, 1948) | Accuracy drops 30-40% by end |
| **Selective** | Can you ignore distractors? | Stroop Effect (Stroop, 1935) | Factual distractors intrude 5x more |
| **Shifting** | Can you switch rules mid-task? | Wisconsin Card Sort (Monsell, 2003) | 60%+ of errors are perseveration |
| **Stimulus-Driven** | Do you notice the unexpected? | Gorilla Experiment (Simons, 1999) | Small models score 0.16 |

Every item is **procedurally generated** from a seed — no static datasets, no contamination risk, ground truth is always unambiguous.

---

## The Human-AI Inversion

In [ ]:
import plotly.graph_objects as go

abilities = ['Capacity', 'Sustained', 'Selective', 'Shifting', 'Stimulus-Driven']

profiles = {
    'Qwen2.5-72B':   {'scores': [0.83, 0.93, 0.84, 0.90, 0.48], 'color': '#4ec9b0'},
    'Llama-3.1-8B':  {'scores': [0.75, 0.81, 0.83, 0.65, 0.40], 'color': '#5b9bd5'},
    'Phi-3.5-mini':  {'scores': [0.22, 0.68, 0.51, 0.61, 0.16], 'color': '#f0c060'},
    'Human (est.)':  {'scores': [0.60, 0.85, 0.95, 0.80, 0.90], 'color': '#e94560'},
}

fig = go.Figure()
for name, p in profiles.items():
    fig.add_trace(go.Scatterpolar(
        r=p['scores'] + [p['scores'][0]], theta=abilities + [abilities[0]],
        fill='toself', name=name, line=dict(color=p['color'], width=2.5), opacity=0.55))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1], tickfont=dict(size=10)),
               angularaxis=dict(tickfont=dict(size=13, color='#e6edf3')), bgcolor='#0d1117'),
    title=dict(text='Cognitive Profile: AI vs Human', font=dict(size=18, color='#e6edf3')),
    paper_bgcolor='#0d1117', font=dict(color='#e6edf3'),
    legend=dict(font=dict(size=12), bgcolor='rgba(0,0,0,0)'),
    width=750, height=600)
fig.show()

**Look at the shapes.** Humans (red) are strong at noticing anomalies but weak at tracking many objects. LLMs show the **exact inverse** — vast context windows but attention dilutes over length.

This isn't a bug. It reflects fundamentally different architectures:
- **Humans:** parallel sensory processing, limited working memory (~4 items)
- **Transformers:** unlimited "working memory" (context window), but softmax dilutes attention over distance

---

## How Difficulty Breaks Models

We don't just test Easy. We have **5 tiers** from Easy to Frontier — designed to break even GPT-4o and Claude.

In [ ]:
from plotly.subplots import make_subplots

diffs = ['Easy', 'Medium', 'Hard', 'Expert', 'Frontier']
tasks_data = {
    'Sustained Attention': {
        'Qwen-72B': [1.0, 1.0, 0.70, 0.96, 1.0],
        'Llama-8B': [1.0, 0.88, 0.58, 0.46, 0.83],
        'Phi-3.5':  [0.90, 0.69, 0.55, 0.58, 0.67],
    },
    'Proactive Interference': {
        'Qwen-72B': [1.0, 1.0, 1.0, 1.0, 1.0],
        'Llama-8B': [1.0, 0.0, 0.50, 0.0, 0.44],
        'Phi-3.5':  [1.0, 0.75, 0.50, 0.50, 0.06],
    },
    'Attention Shifting': {
        'Qwen-72B': [0.92, 0.88, 1.0, 0.85, 0.88],
        'Llama-8B': [0.92, 0.62, 0.60, 0.60, 0.52],
        'Phi-3.5':  [0.92, 0.62, 0.75, 0.45, 0.29],
    },
    'Anomaly Detection': {
        'Qwen-72B': [0.50, 0.60, 0.20, 0.70, 0.40],
        'Llama-8B': [1.0, 0.80, 0.0, 0.20, 0.0],
        'Phi-3.5':  [0.20, 0.0, 0.20, 0.20, 0.20],
    },
}
mc = {'Qwen-72B': '#4ec9b0', 'Llama-8B': '#5b9bd5', 'Phi-3.5': '#f0c060'}

fig = make_subplots(rows=2, cols=2, subplot_titles=list(tasks_data.keys()))
for (task, data), (r, c) in zip(tasks_data.items(), [(1,1),(1,2),(2,1),(2,2)]):
    for model, scores in data.items():
        fig.add_trace(go.Scatter(
            x=diffs, y=scores, mode='lines+markers', name=model,
            line=dict(color=mc[model], width=2.5), marker=dict(size=8),
            showlegend=(r==1 and c==1)), row=r, col=c)

fig.update_layout(height=650, width=900,
    title=dict(text='Performance Collapse Across Difficulty Tiers', font=dict(size=16, color='#e6edf3')),
    paper_bgcolor='#0d1117', plot_bgcolor='#0d1117', font=dict(color='#e6edf3', size=11))
fig.update_yaxes(range=[-0.05, 1.1], gridcolor='#21262d')
fig.update_xaxes(gridcolor='#21262d')
fig.show()

**Proactive Interference** (top right) is devastating: Llama-8B goes from 1.0 at Easy to **0.0** at Expert. Even after 50 updates to the same key, Qwen-72B still tracks the latest value perfectly — an architectural advantage of scale.

**Anomaly Detection** (bottom right) is where even large models struggle. Qwen-72B only scores 0.40 at Frontier — it's so focused on the primary counting task that it misses subtle anomalies.

---

## Not Just *That* Models Fail — *How* They Fail

We don't stop at accuracy. We classify the **type** of error.

In [ ]:
fig = go.Figure()
models_list = ['Qwen-72B', 'Llama-8B', 'Phi-3.5']

fig.add_trace(go.Bar(name='Perseveration (stuck on old rule)',
    x=models_list, y=[2, 5, 8], marker_color='#e74c3c'))
fig.add_trace(go.Bar(name='Attentional Residue (old context bleeds in)',
    x=models_list, y=[1, 3, 5], marker_color='#f39c12'))
fig.add_trace(go.Bar(name='Random Error',
    x=models_list, y=[1, 2, 4], marker_color='#555'))

fig.update_layout(barmode='stack',
    title=dict(text='Rule Shifting: Error Classification', font=dict(size=16, color='#e6edf3')),
    yaxis_title='Error Count',
    paper_bgcolor='#0d1117', plot_bgcolor='#0d1117',
    font=dict(color='#e6edf3'), width=700, height=420,
    legend=dict(font=dict(size=11), bgcolor='rgba(0,0,0,0)'))
fig.update_yaxes(gridcolor='#21262d')
fig.show()

**60-70% of shifting errors are NOT random hallucination.** They are:
- **Perseveration** — the model keeps applying the old rule after being told to switch
- **Attentional Residue** — the model's answer comes from pre-switch context

This maps directly to **causal self-attention**: because all preceding tokens influence generation, the old rule's tokens continue to pull the attention matrix even after the switch instruction.

---

## Overall Scores

In [ ]:
import plotly.graph_objects as go

models_cas = ['Qwen-72B', 'Llama-8B', 'Phi-3.5']
cas_scores = [0.833, 0.685, 0.567]
colors = ['#4ec9b0', '#5b9bd5', '#f0c060']

fig = go.Figure()
fig.add_trace(go.Bar(x=models_cas, y=cas_scores, marker_color=colors,
    text=[f'{s:.3f}' for s in cas_scores], textposition='outside',
    textfont=dict(size=14, color='#e6edf3')))

fig.update_layout(
    title=dict(text='Cognitive Attention Score (CAS) by Model', font=dict(size=16, color='#e6edf3')),
    yaxis=dict(title='CAS Score', range=[0, 1.05]),
    paper_bgcolor='#0d1117', plot_bgcolor='#0d1117',
    font=dict(color='#e6edf3'), width=600, height=420)
fig.update_yaxes(gridcolor='#21262d')
fig.show()

CAS (Cognitive Attention Score) is a weighted composite across all task types. Higher = better attention.

The gap between Qwen-72B (0.833) and Phi-3.5 (0.567) confirms that attention abilities scale with model size — but even the largest model has blind spots.

---

## Why Attention Fails: The Architecture Connection

| What the Model Does Wrong | Why It Happens (Transformer Architecture) | Our Metric |
|---------------------------|------------------------------------------|------------|
| Misses targets in the middle of long text | **Softmax dilution** — attention weights spread thin over distance (RoPE decay) | Position-stratified accuracy (U-curve) |
| Keeps using the old rule after a switch | **Causal attention** — all preceding tokens influence generation, old rule context persists | Perseveration rate + residue rate |
| Can't ignore factual distractors | **MLP pre-training priors** overpower in-context attention heads | Stroop error rate: factual vs nonsense |
| Tracks only 3-4 objects before failing | **Fixed attention heads** — each layer has limited parallel capacity | Capacity curve breakpoint |
| Misses anomalies during counting tasks | **No surplus attention** — all heads allocated to primary task | Dual-task score under load |

---

## The Scorecard

| Model | CAS Score | Strongest Ability | Weakest Ability |
|-------|-----------|-------------------|-----------------|
| **Qwen2.5-72B** | 0.833 | Interference (1.0) | Anomaly (0.48) |
| **Llama-3.1-8B** | 0.685 | Flanker (1.0) | Inhibition of Return (0.27) |
| **Phi-3.5-mini** | 0.567 | Flanker (1.0) | Anomaly (0.16) |

| Strongest Discriminators | Spread | What it reveals |
|--------------------------|--------|-----------------|
| Inhibition of Return | 0.73 | Return-to-context accuracy varies wildly |
| Proactive Interference | 0.61 | Larger models handle state updates better |
| Capacity (Thread Tracking) | 0.41 | Object tracking scales with model size |

---

## References

- Cherry, E. C. (1953). Some experiments on the recognition of speech, with one and with two ears. *JASA*, 25(5), 975-979.
- Mackworth, N. H. (1948). The breakdown of vigilance during prolonged visual search. *QJEP*, 1(1), 6-21.
- Monsell, S. (2003). Task switching. *Trends in Cognitive Sciences*, 7(3), 134-140.
- Posner, M. I., & Petersen, S. E. (1990). The attention system of the human brain. *Annual Review of Neuroscience*, 13, 25-42.
- Pylyshyn, Z. W., & Storm, R. W. (1988). Tracking multiple independent targets. *Spatial Vision*, 3(3), 179-197.
- Simons, D. J., & Chabris, C. F. (1999). Gorillas in our midst. *Perception*, 28(9), 1059-1074.
- Sohlberg, M. M., & Mateer, C. A. (1987). Effectiveness of an attention-training program. *JCEN*, 9(2), 117-130.
- Stroop, J. R. (1935). Studies of interference in serial verbal reactions. *JEP*, 18(6), 643-662.
- Wang, C., & Sun, J. V. (2025). PI-LLM: Proactive interference reveals working memory limits in LLMs. *ICML Workshop*. arXiv:2506.08184.

---

*CogAttention — every instance procedurally generated, every score grounded in cognitive science, every error mechanistically explained.*